In [25]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [26]:
df=pd.read_csv(r"C:\Users\pc\Downloads\archive (19)\Buy_Now_Pay_Later_BNPL_CreditRisk_Dataset.csv")
df.head()

,user_id,age,employment_type,monthly_income,credit_score,purchase_amount,product_category,bnpl_installments,repayment_delay_days,missed_payments,default_flag,app_usage_frequency,location,transaction_date,debt_to_income_ratio,risk_score,customer_segment
0,1,56,Salaried,68529.50,552,5000.00,Electronics,12,13,1,0,8.49,Australia,2023-06-10,0.072961,165.2,Medium Risk
1,2,19,Student,7247.85,300,1073.23,Fashion,12,13,1,0,3.09,USA,2024-10-07,0.148076,266.0,High Risk
2,3,20,Self-Employed,41582.26,471,5000.00,Electronics,3,19,2,0,3.33,Australia,2023-04-05,0.120244,229.6,High Risk
3,4,21,Salaried,14423.46,300,4076.83,Sports,6,18,5,1,5.86,Germany,2023-06-24,0.282653,356.0,High Risk
4,5,43,Salaried,42845.50,512,5000.00,Electronics,9,0,0,0,7.36,India,2024-10-19,0.116698,135.2,High Risk


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10345 entries, 0 to 10344
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   user_id               10345 non-null  int64  
 1   age                   10345 non-null  int64  
 2   employment_type       10345 non-null  object 
 3   monthly_income        10345 non-null  float64
 4   credit_score          10345 non-null  int64  
 5   purchase_amount       10345 non-null  float64
 6   product_category      10345 non-null  object 
 7   bnpl_installments     10345 non-null  int64  
 8   repayment_delay_days  10345 non-null  int64  
 9   missed_payments       10345 non-null  int64  
 10  default_flag          10345 non-null  int64  
 11  app_usage_frequency   10345 non-null  float64
 12  location              10345 non-null  object 
 13  transaction_date      10345 non-null  object 
 14  debt_to_income_ratio  10345 non-null  float64
 15  risk_score         

In [28]:
df.nunique()

user_id                 10345
age                        42
employment_type             4
monthly_income           9984
credit_score              534
purchase_amount          4011
product_category            5
bnpl_installments           4
repayment_delay_days       33
missed_payments             8
default_flag                2
app_usage_frequency       901
location                    6
transaction_date          365
debt_to_income_ratio    10327
risk_score                876
customer_segment            3
dtype: int64

In [29]:
df['customer_segment'].value_counts()

customer_segment
High Risk      7569
Medium Risk    2200
Low Risk        576
Name: count, dtype: int64

In [30]:
df.groupby('age')['customer_segment'].value_counts()

age  customer_segment
18   High Risk           199
     Medium Risk          23
19   High Risk           213
     Medium Risk          31
20   High Risk           234
                        ... 
58   Medium Risk          57
     Low Risk             20
59   High Risk           158
     Medium Risk          65
     Low Risk             28
Name: count, Length: 121, dtype: int64

In [31]:
df.groupby('location')['customer_segment'].value_counts()

location   customer_segment
Australia  High Risk           1250
           Medium Risk          368
           Low Risk             104
Canada     High Risk           1293
           Medium Risk          344
           Low Risk             103
Germany    High Risk           1309
           Medium Risk          368
           Low Risk              83
India      High Risk           1247
           Medium Risk          389
           Low Risk              87
UK         High Risk           1201
           Medium Risk          369
           Low Risk              94
USA        High Risk           1269
           Medium Risk          362
           Low Risk             105
Name: count, dtype: int64

In [32]:
df.groupby('employment_type')['customer_segment'].value_counts()

employment_type  customer_segment
Salaried         High Risk           1580
                 Medium Risk          910
                 Low Risk             142
Self-Employed    High Risk           1074
                 Medium Risk         1045
                 Low Risk             431
Student          High Risk           2452
                 Medium Risk           77
                 Low Risk               2
Unemployed       High Risk           2463
                 Medium Risk          168
                 Low Risk               1
Name: count, dtype: int64

In [33]:
df.isna().sum()

user_id                 0
age                     0
employment_type         0
monthly_income          0
credit_score            0
purchase_amount         0
product_category        0
bnpl_installments       0
repayment_delay_days    0
missed_payments         0
default_flag            0
app_usage_frequency     0
location                0
transaction_date        0
debt_to_income_ratio    0
risk_score              0
customer_segment        0
dtype: int64

In [34]:
df.duplicated().sum()

np.int64(0)

In [35]:
df=df.drop(columns=['user_id','monthly_income','purchase_amount','transaction_date','risk_score'])

In [36]:
object_cols=df.select_dtypes(object).columns.tolist()
object_cols.remove('customer_segment')

In [37]:
X=df.drop('customer_segment',axis=1)
Y=df['customer_segment'].map({
    'Low Risk': 0,
    'Medium Risk': 1,
    'High Risk':2
})

In [38]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.2)

In [39]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
ohe=OneHotEncoder()
scaler=StandardScaler()
x_train_encoded=ohe.fit_transform(x_train[object_cols])
x_test_encoded=ohe.transform(x_test[object_cols])

x_train_ohe_df = pd.DataFrame.sparse.from_spmatrix(
    x_train_encoded,
    columns=ohe.get_feature_names_out(object_cols),
    index=x_train.index
)

x_test_ohe_df = pd.DataFrame.sparse.from_spmatrix(
    x_test_encoded,
    columns=ohe.get_feature_names_out(object_cols),
    index=x_test.index
)

# Drop original categorical columns
x_train_num = x_train.drop(columns=object_cols)
x_test_num = x_test.drop(columns=object_cols)

x_train_num_scaled=scaler.fit_transform(x_train_num)
x_test_num_scaled=scaler.transform(x_test_num)

# Convert scaled arrays back to DataFrames
x_train_num_scaled_df = pd.DataFrame(
    x_train_num_scaled,
    columns=x_train_num.columns,
    index=x_train.index
)

x_test_num_scaled_df = pd.DataFrame(
    x_test_num_scaled,
    columns=x_test_num.columns,
    index=x_test.index
)

# Concatenate
x_train_final = pd.concat([x_train_num_scaled_df, x_train_ohe_df], axis=1)
x_test_final = pd.concat([x_test_num_scaled_df, x_test_ohe_df], axis=1)

In [40]:
from xgboost import XGBClassifier

In [41]:
xgc=XGBClassifier()
xgc.fit(x_train_final,y_train)
y_pred=xgc.predict(x_test_final)
from sklearn.metrics import confusion_matrix
cm=confusion_matrix(y_test,y_pred)
print(cm)

C:\Users\pc\anaconda3\Lib\site-packages\xgboost\data.py:400: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")


[[ 104    0    0]
 [   3  433    6]
 [   0    9 1514]]


C:\Users\pc\anaconda3\Lib\site-packages\xgboost\data.py:400: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")


In [42]:
##Balancing the dataset by oversampling using smote
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

x_resampled, y_resampled = smote.fit_resample(x_train_final, y_train)

C:\Users\pc\anaconda3\Lib\site-packages\sklearn\utils\validation.py:921: UserWarning: pandas.DataFrame with sparse columns found.It will be converted to a dense numpy array.
  warnings.warn(


In [43]:
xgc.fit(x_resampled, y_resampled)
y_pred1=xgc.predict(x_test_final)
cm1=confusion_matrix(y_test,y_pred1)
print(cm1)

C:\Users\pc\anaconda3\Lib\site-packages\xgboost\data.py:400: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")


[[ 104    0    0]
 [   0  437    5]
 [   0   13 1510]]


C:\Users\pc\anaconda3\Lib\site-packages\xgboost\data.py:400: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")


In [44]:
##Hyperparameter tuning
xgb_params = {
    "max_depth":[3,5,6,7],
    "learning_rate": [0.01,0.05,0.09,0.1],
    "n_estimators": [100,200,300]
}

In [45]:
from sklearn.model_selection import RandomizedSearchCV
clf=RandomizedSearchCV(xgc,xgb_params,cv=3)
clf.fit(x_resampled, y_resampled)
clf.best_params_

C:\Users\pc\anaconda3\Lib\site-packages\xgboost\data.py:400: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")
C:\Users\pc\anaconda3\Lib\site-packages\xgboost\data.py:400: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")
C:\Users\pc\anaconda3\Lib\site-packages\xgboost\data.py:400: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")
C:\Users\pc\anaconda3\Lib\site-packages\xgboost\data.py:400: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")
C:\Users\pc\anaconda3\Lib\site-packages\xgboost\data.py:400: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")
C:\Users\pc\ana

{'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.01}

In [46]:
xg=XGBClassifier(n_estimators= 200, max_depth= 5, learning_rate= 0.01)
xg.fit(x_resampled, y_resampled)
y_pred3=xg.predict(x_test_final)
cm3=confusion_matrix(y_test,y_pred3)
print(cm3)

C:\Users\pc\anaconda3\Lib\site-packages\xgboost\data.py:400: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")


[[ 104    0    0]
 [   0  440    2]
 [   0   14 1509]]


C:\Users\pc\anaconda3\Lib\site-packages\xgboost\data.py:400: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")
